<a href="https://colab.research.google.com/github/kaykesm/MLCB_CK_2026/blob/main/AULA_05/exercicios03e04_aula05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q pandas numpy scikit-learn nltk spacy gensim


In [24]:
!python -m spacy download pt_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 57.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [25]:
import re
import numpy as np
import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from gensim.models import FastText

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

nltk.download("stopwords", quiet=True)

stop_words_pt = set(stopwords.words("portuguese"))

nlp = spacy.load("pt_core_news_sm")

print("Ambiente preparado com sucesso!")


Ambiente preparado com sucesso!


In [26]:
df = pd.read_csv("sac_moveis_ac2_treino.csv")

print("Dataset carregado!")
print("Quantidade de mensagens:", len(df))
print("Colunas:", df.columns.tolist())

display(df.head())


Dataset carregado!
Quantidade de mensagens: 80
Colunas: ['mensagem', 'intencao']


,mensagem,intencao
0,Meu pedido está atrasado,logistica_entregas
1,Quero devolver este sofá que chegou com rasgo,trocas_devolucoes
2,Qual o prazo de entrega do sofá que comprei?,logistica_entregas
3,Quero consultar o status da entrega,logistica_entregas
4,Recebi um armário com peças quebradas e quero ...,trocas_devolucoes


In [27]:
def limpar_e_lemmatizar(texto):
    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens_filtrados = []

    for token in doc:

        if token.is_space:
            continue

        if token.text in stop_words_pt:
            continue

        if len(token.text) <= 1:
            continue

        tokens_filtrados.append(token.lemma_)

    return " ".join(tokens_filtrados)


In [28]:
df["texto_normalizado"] = df["mensagem"].apply(
    limpar_e_lemmatizar
)

display(
    df[["mensagem", "texto_normalizado", "intencao"]].head(10)
)


,mensagem,texto_normalizado,intencao
0,Meu pedido está atrasado,pedido atrasado,logistica_entregas
1,Quero devolver este sofá que chegou com rasgo,querer devolver sofá chegar rasgo,trocas_devolucoes
2,Qual o prazo de entrega do sofá que comprei?,prazo entrega sofá comprei,logistica_entregas
3,Quero consultar o status da entrega,querer consultar status entrega,logistica_entregas
4,Recebi um armário com peças quebradas e quero ...,recebi armário peça quebrar querer devolver,trocas_devolucoes
5,Como faço para rastrear meu pedido?,fazer rastrear pedido,logistica_entregas
6,O produto chegou diferente do que comprei,produto chegar diferente comprei,trocas_devolucoes
7,Vocês têm promoção de sofá?,ter promoção sofá,vendas_orcamento
8,Preciso devolver a cadeira de escritório com d...,preciso devolver cadeira escritório defeito,trocas_devolucoes
9,"Meu sofá chegou rasgado, como faço a troca?",sofá chegar rasgado fazer troca,trocas_devolucoes


In [29]:
corpus_tokenizado = [
    texto.split()
    for texto in df["texto_normalizado"]
]

print("Quantidade de mensagens:", len(corpus_tokenizado))
print("\nPrimeira mensagem:")
print(corpus_tokenizado[0])


Quantidade de mensagens: 80

Primeira mensagem:
['pedido', 'atrasado']


In [30]:
modelo_fasttext = FastText(
    sentences=corpus_tokenizado,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=1
)

print("Modelo FastText treinado com sucesso!")
print("Dimensão dos vetores:", modelo_fasttext.vector_size)


Modelo FastText treinado com sucesso!
Dimensão dos vetores: 50


In [31]:
def obter_vetor_frase(frase, modelo):
    """
    Transforma uma frase em um vetor denso utilizando
    FastText + Mean Pooling.
    """

    palavras = frase.split()

    vetores = []

    for palavra in palavras:
        vetor = modelo.wv[palavra]
        vetores.append(vetor)

    if len(vetores) == 0:
        return np.zeros(modelo.vector_size)

    vetor_medio = np.mean(vetores, axis=0)

    return vetor_medio


In [32]:
X_vetores = np.array([
    obter_vetor_frase(texto, modelo_fasttext)
    for texto in df["texto_normalizado"]
])

y = df["intencao"].values

print("Dimensão de X_vetores:")
print(X_vetores.shape)

print("\nDimensão de y:")
print(y.shape)


Dimensão de X_vetores:
(80, 50)

Dimensão de y:
(80,)


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vetores,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dados de treinamento:", X_train.shape)
print("Dados de teste:", X_test.shape)


Dados de treinamento: (64, 50)
Dados de teste: (16, 50)


In [34]:
modelo = LogisticRegression(max_iter=1000)

modelo.fit(X_train, y_train)

print("Modelo de Regressão Logística treinado com sucesso!")


Modelo de Regressão Logística treinado com sucesso!


In [35]:
y_pred = modelo.predict(X_test)

print("Previsões realizadas com sucesso!")
print("\nPrimeiras previsões:")
print(y_pred[:10])


Previsões realizadas com sucesso!

Primeiras previsões:
['vendas_orcamento' 'vendas_orcamento' 'logistica_entregas'
 'suporte_tecnico' 'logistica_entregas' 'trocas_devolucoes'
 'vendas_orcamento' 'trocas_devolucoes' 'trocas_devolucoes'
 'trocas_devolucoes']


In [36]:
print(classification_report(y_test, y_pred))


                    precision    recall  f1-score   support

logistica_entregas       1.00      0.75      0.86         4
   suporte_tecnico       1.00      0.75      0.86         4
 trocas_devolucoes       0.67      1.00      0.80         4
  vendas_orcamento       0.50      0.50      0.50         4

          accuracy                           0.75        16
         macro avg       0.79      0.75      0.75        16
      weighted avg       0.79      0.75      0.75        16



In [37]:
def classificar_mensagem(
    mensagem,
    modelo,
    modelo_embedding,
    limiar=0.50
):
    # Pré-processar a mensagem
    mensagem_normalizada = limpar_e_lemmatizar(mensagem)

    # Transformar a mensagem em vetor
    vetor = obter_vetor_frase(
        mensagem_normalizada,
        modelo_embedding
    )

    # Adicionar dimensão para o modelo
    vetor = vetor.reshape(1, -1)

    # Obter probabilidades
    probabilidades = modelo.predict_proba(vetor)[0]

    # Encontrar a maior probabilidade
    indice = np.argmax(probabilidades)

    confianca = probabilidades[indice]

    intencao = modelo.classes_[indice]

    # Aplicar o fallback
    if confianca >= limiar:
        return intencao, confianca
    else:
        return "FALLBACK_HUMANO", confianca


In [38]:
testes = [
    "quero devolver meu sofá",
    "como faço para realizar a devolução?",
    "cadê meu pedido?",
    "meu pedido nao chego",
    "qual é a previsão do tempo?"
]

for mensagem in testes:
    intencao, confianca = classificar_mensagem(
        mensagem,
        modelo,
        modelo_fasttext,
        limiar=0.50
    )

    print("Mensagem:", mensagem)
    print("Intenção:", intencao)
    print(f"Confiança: {confianca:.2%}")
    print("-" * 50)


Mensagem: quero devolver meu sofá
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
--------------------------------------------------
Mensagem: como faço para realizar a devolução?
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
--------------------------------------------------
Mensagem: cadê meu pedido?
Intenção: FALLBACK_HUMANO
Confiança: 25.01%
--------------------------------------------------
Mensagem: meu pedido nao chego
Intenção: FALLBACK_HUMANO
Confiança: 25.01%
--------------------------------------------------
Mensagem: qual é a previsão do tempo?
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
--------------------------------------------------


In [39]:
for mensagem in testes:
    mensagem_normalizada = limpar_e_lemmatizar(mensagem)

    vetor = obter_vetor_frase(
        mensagem_normalizada,
        modelo_fasttext
    ).reshape(1, -1)

    probabilidades = modelo.predict_proba(vetor)[0]

    print("\nMensagem:", mensagem)

    for classe, probabilidade in zip(
        modelo.classes_,
        probabilidades
    ):
        print(f"{classe}: {probabilidade:.2%}")



Mensagem: quero devolver meu sofá
logistica_entregas: 25.00%
suporte_tecnico: 25.00%
trocas_devolucoes: 25.00%
vendas_orcamento: 25.00%

Mensagem: como faço para realizar a devolução?
logistica_entregas: 25.00%
suporte_tecnico: 25.00%
trocas_devolucoes: 25.00%
vendas_orcamento: 25.00%

Mensagem: cadê meu pedido?
logistica_entregas: 25.01%
suporte_tecnico: 25.00%
trocas_devolucoes: 25.00%
vendas_orcamento: 25.00%

Mensagem: meu pedido nao chego
logistica_entregas: 25.01%
suporte_tecnico: 24.99%
trocas_devolucoes: 25.00%
vendas_orcamento: 25.00%

Mensagem: qual é a previsão do tempo?
logistica_entregas: 24.99%
suporte_tecnico: 25.00%
trocas_devolucoes: 25.00%
vendas_orcamento: 25.00%


In [40]:
limiares = [0.25, 0.50, 0.75]

mensagem = "quero devolver meu sofá"

for limiar in limiares:
    intencao, confianca = classificar_mensagem(
        mensagem,
        modelo,
        modelo_fasttext,
        limiar=limiar
    )

    print(f"Limiar: {limiar:.0%}")
    print(f"Intenção: {intencao}")
    print(f"Confiança: {confianca:.2%}")
    print("-" * 40)


Limiar: 25%
Intenção: trocas_devolucoes
Confiança: 25.00%
----------------------------------------
Limiar: 50%
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
----------------------------------------
Limiar: 75%
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
----------------------------------------


In [ ]:
# Exercício 4 — Laboratório Comparativo: Regressão Logística × KNN


In [41]:
modelo_logistico = LogisticRegression(
    max_iter=1000
)

modelo_knn = KNeighborsClassifier(
    n_neighbors=3
)

print("Modelos criados com sucesso!")


NameError: name 'KNeighborsClassifier' is not defined

In [42]:
from sklearn.neighbors import KNeighborsClassifier


In [43]:
modelo_logistico = LogisticRegression(
    max_iter=1000
)

modelo_knn = KNeighborsClassifier(
    n_neighbors=3
)

print("Modelos criados com sucesso!")


Modelos criados com sucesso!


In [44]:
modelo_logistico.fit(X_train, y_train)

modelo_knn.fit(X_train, y_train)

print("Regressão Logística treinada!")
print("KNN treinado!")


Regressão Logística treinada!
KNN treinado!


In [45]:
y_pred_logistico = modelo_logistico.predict(X_test)

y_pred_knn = modelo_knn.predict(X_test)

print("Previsões realizadas com sucesso!")

print("\nPrevisões - Regressão Logística:")
print(y_pred_logistico)

print("\nPrevisões - KNN:")
print(y_pred_knn)


Previsões realizadas com sucesso!

Previsões - Regressão Logística:
['vendas_orcamento' 'vendas_orcamento' 'logistica_entregas'
 'suporte_tecnico' 'logistica_entregas' 'trocas_devolucoes'
 'vendas_orcamento' 'trocas_devolucoes' 'trocas_devolucoes'
 'trocas_devolucoes' 'trocas_devolucoes' 'suporte_tecnico'
 'trocas_devolucoes' 'suporte_tecnico' 'logistica_entregas'
 'vendas_orcamento']

Previsões - KNN:
['logistica_entregas' 'vendas_orcamento' 'logistica_entregas'
 'vendas_orcamento' 'logistica_entregas' 'suporte_tecnico'
 'logistica_entregas' 'trocas_devolucoes' 'suporte_tecnico'
 'trocas_devolucoes' 'trocas_devolucoes' 'suporte_tecnico'
 'trocas_devolucoes' 'trocas_devolucoes' 'trocas_devolucoes'
 'logistica_entregas']


In [46]:
# Regressão Logística
accuracy_logistico = accuracy_score(y_test, y_pred_logistico)
precision_logistico = precision_score(
    y_test,
    y_pred_logistico,
    average="weighted"
)
recall_logistico = recall_score(
    y_test,
    y_pred_logistico,
    average="weighted"
)
f1_logistico = f1_score(
    y_test,
    y_pred_logistico,
    average="weighted"
)

# KNN
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(
    y_test,
    y_pred_knn,
    average="weighted"
)
recall_knn = recall_score(
    y_test,
    y_pred_knn,
    average="weighted"
)
f1_knn = f1_score(
    y_test,
    y_pred_knn,
    average="weighted"
)

print("REGRESSÃO LOGÍSTICA")
print(f"Accuracy:  {accuracy_logistico:.2%}")
print(f"Precision: {precision_logistico:.2%}")
print(f"Recall:    {recall_logistico:.2%}")
print(f"F1:        {f1_logistico:.2%}")

print("\nKNN")
print(f"Accuracy:  {accuracy_knn:.2%}")
print(f"Precision: {precision_knn:.2%}")
print(f"Recall:    {recall_knn:.2%}")
print(f"F1:        {f1_knn:.2%}")


REGRESSÃO LOGÍSTICA
Accuracy:  75.00%
Precision: 79.17%
Recall:    75.00%
F1:        75.36%

KNN
Accuracy:  56.25%
Precision: 52.50%
Recall:    56.25%
F1:        52.14%


In [47]:
resultado_comparacao = pd.DataFrame({
    "Modelo": [
        "Regressão Logística",
        "KNN"
    ],
    "Accuracy": [
        accuracy_logistico,
        accuracy_knn
    ],
    "Precision": [
        precision_logistico,
        precision_knn
    ],
    "Recall": [
        recall_logistico,
        recall_knn
    ],
    "F1": [
        f1_logistico,
        f1_knn
    ]
})

resultado_comparacao.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1": "{:.2%}"
})


,Modelo,Accuracy,Precision,Recall,F1
0,Regressão Logística,75.00%,79.17%,75.00%,75.36%
1,KNN,56.25%,52.50%,56.25%,52.14%
